In [0]:
%python

# Databricks notebook source
# MAGIC %md
# MAGIC # Claude API Setup -- Foundation Utilities
# MAGIC
# MAGIC This notebook is imported by every other AI module (`02_business_intelligence_assistant.py`,
# MAGIC `03_conversation_classifier.py`, `04_hybrid_bot_prototype.py`) via `%run ./01_claude_api_setup`.
# MAGIC It provides one thing: a `call_claude(prompt, ...)` function that's safe to call from anywhere
# MAGIC without each module reimplementing rate limiting, retries, and cost tracking.
# MAGIC
# MAGIC **What's in here, and why each piece exists:**
# MAGIC - Secret-based API key (never hardcoded in a notebook)
# MAGIC - A connectivity test (confirms outbound access works on this cluster before you build on top of it)
# MAGIC - Rate limiting (prevents hammering the API faster than it allows)
# MAGIC - In-memory response caching (skip paying twice for an identical prompt in the same session)
# MAGIC - Retry with exponential backoff (transient errors shouldn't kill a whole batch run)
# MAGIC - A cost estimator (check the bill *before* running something against 170K conversations)

# COMMAND ----------

# MAGIC %md
# MAGIC ## SDK: cluster library, not notebook-scoped `%pip`
# MAGIC
# MAGIC `anthropic` needs to be installed as a **cluster library**, not via `%pip install` +
# MAGIC `dbutils.library.restartPython()` in this cell. Reason: this notebook gets pulled into
# MAGIC other notebooks via `%run ./01_claude_api_setup` (from `02_`, `03_`, `04_`), and
# MAGIC `restartPython()` resets the *entire* interpreter for whatever notebook is running --
# MAGIC including the caller. In a `%run` chain that silently breaks everything defined after
# MAGIC the restart, which is exactly the "`call_claude` is not defined" failure this replaced.
# MAGIC
# MAGIC One-time setup per cluster: **Compute -> your cluster -> Libraries tab -> Install New ->
# MAGIC PyPI -> package `anthropic`**. Takes effect once the cluster restarts; after that every
# MAGIC notebook attached to it has `anthropic` available with no per-notebook install step.

# COMMAND ----------

try:
    import anthropic
except ImportError:
    raise ImportError(
        "anthropic is not installed on this cluster. Go to Compute -> your cluster -> "
        "Libraries -> Install New -> PyPI -> 'anthropic', then restart the cluster and "
        "re-run this notebook. Do not fix this with %pip install here -- see the note above."
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## API key via Databricks Secrets
# MAGIC
# MAGIC **Why not just paste the key in a variable?** If this notebook is ever exported, shared, or
# MAGIC pushed to a public repo, a hardcoded key leaks with it. `dbutils.secrets.get()` pulls the value
# MAGIC at runtime from an encrypted store instead -- the key itself never appears in the notebook source.
# MAGIC
# MAGIC Assumes you've already run (from the Databricks CLI, one-time setup):
# MAGIC ```
# MAGIC databricks secrets create-scope --scope claude-api
# MAGIC databricks secrets put-secret claude-api anthropic-key
# MAGIC ```

# COMMAND ----------

SECRET_SCOPE = "claude-api"
SECRET_KEY = "anthropic-key"

# Pricing changes over time and this notebook shouldn't need to change every time it does --
# CLAUDE_MODEL is the one place to swap models.
CLAUDE_MODEL = "claude-sonnet-5"

CLAUDE_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

print(f"Client initialized for model: {CLAUDE_MODEL}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Connectivity test
# MAGIC
# MAGIC Run this **before** building anything else on top of this notebook. This is the cheapest
# MAGIC possible call (1-token response) -- if this fails with a network error rather than an API
# MAGIC error, it means this cluster can't reach the Anthropic API at all (the serverless/trial-tier
# MAGIC egress issue covered earlier), and no amount of debugging the code below will fix that --
# MAGIC you'd need to switch to the classic cluster instead.

# COMMAND ----------

try:
    _test_response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Reply with exactly one word: OK"}],
    )
    print("Connectivity OK -- response:", _test_response.content[0].text.strip())
except anthropic.APIConnectionError as e:
    print("NETWORK-LEVEL FAILURE -- this cluster cannot reach the Anthropic API.")
    print("This is not a code bug. Switch to the classic cluster (not serverless) and retry.")
    print("Details:", e)
except anthropic.AuthenticationError as e:
    print("AUTH FAILURE -- API key is missing/invalid. Check the secret scope/key names above.")
    print("Details:", e)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Rate limiting
# MAGIC
# MAGIC **Why:** the API enforces requests-per-minute and tokens-per-minute limits. Exceeding them
# MAGIC doesn't crash your job politely -- it returns 429 errors mid-batch. A simple minimum-interval
# MAGIC gate between calls avoids hitting that ceiling in the first place.
# MAGIC
# MAGIC **Trade-off:** too conservative (a long interval) makes a 500-conversation classification run
# MAGIC take forever; too aggressive and you still get throttled. `min_interval_seconds=1.0` is a safe
# MAGIC starting point for a personal-tier account -- tighten it once you know your actual rate limit
# MAGIC from the API response headers.

# COMMAND ----------

import time


class RateLimiter:
    def __init__(self, min_interval_seconds: float = 1.0):
        self.min_interval = min_interval_seconds
        self._last_call = 0.0

    def wait(self):
        elapsed = time.time() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.time()


rate_limiter = RateLimiter(min_interval_seconds=1.0)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Retry with exponential backoff
# MAGIC
# MAGIC **Why:** rate-limit errors (429) and server errors (5xx) are usually transient -- retrying
# MAGIC after a short wait often succeeds. Client errors (400, 401, etc.) are not transient -- retrying
# MAGIC a malformed request just wastes time and money, so those raise immediately instead.
# MAGIC
# MAGIC Backoff doubles each attempt (2s, 4s, 8s) so a struggling API gets progressively more room
# MAGIC instead of being retried at a constant, possibly-still-too-fast rate.

# COMMAND ----------

def _call_with_retry(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024,
                      max_retries: int = 3, **kwargs):
    last_error = None
    for attempt in range(max_retries):
        try:
            rate_limiter.wait()
            return client.messages.create(
                model=model,
                max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
                **kwargs,
            )
        except anthropic.RateLimitError as e:
            wait_time = 2 ** attempt * 2
            print(f"Rate limited -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait_time)
            last_error = e
        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait_time = 2 ** attempt * 2
                print(f"Server error {e.status_code} -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
                last_error = e
            else:
                raise  # 4xx errors won't fix themselves on retry
    raise last_error

# COMMAND ----------

# MAGIC %md
# MAGIC ## In-memory caching
# MAGIC
# MAGIC **Why:** if a notebook cell gets rerun (debugging, a crash partway through a loop), identical
# MAGIC prompts shouldn't be billed twice. The cache key includes the model and `max_tokens` so changing
# MAGIC either one is treated as a different request.
# MAGIC
# MAGIC **Limitation:** this is a plain Python dict -- it lives only as long as the cluster session.
# MAGIC It resets on cluster restart. That's an acceptable trade-off for a portfolio project; a
# MAGIC production version would persist this to a Delta table instead.

# COMMAND ----------

import hashlib
import json

_response_cache = {}


def _cache_key(prompt: str, model: str, **kwargs) -> str:
    payload = json.dumps({"prompt": prompt, "model": model, **kwargs}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()

# COMMAND ----------

# MAGIC %md
# MAGIC ## `call_claude()` -- the function every other module actually uses
# MAGIC
# MAGIC Combines caching, rate limiting, and retry into one call. Returns `(text, was_cached)` so
# MAGIC calling code can log/print cache hit rates if it wants to.

# COMMAND ----------

def call_claude(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024, **kwargs):
    """
    Call Claude with caching + rate limiting + retry already handled.

    Returns:
        (response_text: str, was_cached: bool)
    """
    key = _cache_key(prompt, model, max_tokens=max_tokens, **kwargs)

    if key in _response_cache:
        return _response_cache[key], True

    response = _call_with_retry(prompt, model=model, max_tokens=max_tokens, **kwargs)
    text = response.content[0].text
    _response_cache[key] = text
    return text, False

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cost estimator
# MAGIC
# MAGIC **Why:** the classifier module (Module 2) will process a sample of conversations, not all
# MAGIC 170K -- this function is how you check the cost of that sample *before* running it, not after.
# MAGIC
# MAGIC Pricing below is per Anthropic's published rates as of August 2026 and is a promotional rate
# MAGIC that reverts to standard pricing after August 31, 2026 -- **verify current pricing at
# MAGIC https://platform.claude.com/docs/en/about-claude/pricing before trusting this for a real budget
# MAGIC decision**, since rates change and this table won't update itself.

# COMMAND ----------

# $ per million tokens (input, output) -- update if pricing changes
PRICING_PER_MILLION_TOKENS = {
    "claude-sonnet-5": {"input": 2.00, "output": 10.00},   # promotional rate through Aug 31, 2026
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-opus-5": {"input": 15.00, "output": 75.00},    # approximate -- verify before use
}


def estimate_cost(num_calls: int, avg_input_tokens: int, avg_output_tokens: int,
                   model: str = CLAUDE_MODEL) -> float:
    """
    Rough cost estimate for a batch of calls. Use this before running anything against
    more than a handful of rows -- e.g. estimate_cost(500, 300, 150) for a 500-conversation
    classification sample.
    """
    if model not in PRICING_PER_MILLION_TOKENS:
        raise ValueError(f"No pricing entry for '{model}' -- add one to PRICING_PER_MILLION_TOKENS first")

    rates = PRICING_PER_MILLION_TOKENS[model]
    input_cost = (num_calls * avg_input_tokens / 1_000_000) * rates["input"]
    output_cost = (num_calls * avg_output_tokens / 1_000_000) * rates["output"]
    total = input_cost + output_cost

    print(f"Estimated cost for {num_calls:,} calls on {model}:")
    print(f"  Input:  {num_calls * avg_input_tokens:,} tokens -> ${input_cost:.2f}")
    print(f"  Output: {num_calls * avg_output_tokens:,} tokens -> ${output_cost:.2f}")
    print(f"  Total:  ${total:.2f}")
    return total

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ready for the other modules
# MAGIC
# MAGIC `02_business_intelligence_assistant.py`, `03_conversation_classifier.py`, and
# MAGIC `04_hybrid_bot_prototype.py` should each start with `%run ./01_claude_api_setup`, then use
# MAGIC `call_claude(prompt)` directly. Run `estimate_cost(...)` before any loop that calls it more
# MAGIC than a few dozen times.

# COMMAND ----------

print("=" * 70)
print("01_claude_api_setup ready")
print("=" * 70)
print(f"Model: {CLAUDE_MODEL}")
print("Available: call_claude(prompt), estimate_cost(num_calls, avg_in, avg_out)")
print("Cache size this session:", len(_response_cache))

